# fiber-mosaic quickstart

Build per-color fiber photometry recordings, read fluorescence with the fiber-native API, and bundle multiple colors into a single experiment handle.

Layering used by the library:

- **fiber** &rarr; one channel of a recording
- **color** &rarr; one `BaseFiberPhotometryExtractor` (each color has its own timebase)
- **experiment** &rarr; one `FiberPhotometryRecordingGroup` holding all colors

In [1]:
# matplotlib is only needed for the plotting cell below
# %pip install matplotlib

import numpy as np

from fiber_mosaic import (
    BaseFiberPhotometryExtractor,
    FiberPhotometryRecordingGroup,
    NumpyFiberPhotometrySegment,
)

ModuleNotFoundError: No module named 'fiber_mosaic'

## 1. Synthesize a single-color recording

Three fibers, 5 seconds at 100 Hz. In a real experiment the array would come from your acquisition file via a subclass of `BaseFiberPhotometryFileExtractor`.

In [ ]:
sampling_rate = 100.0  # Hz
n_samples = 500
fiber_ids = ["VTA_left", "VTA_right", "NAc"]

t = np.arange(n_samples) / sampling_rate
rng = np.random.default_rng(0)

green_traces = np.column_stack(
    [
        1.0 + 0.20 * np.sin(2 * np.pi * 0.5 * t),
        1.0 + 0.30 * np.sin(2 * np.pi * 0.8 * t + 1.0),
        1.0 + 0.10 * np.sin(2 * np.pi * 1.2 * t + 2.0),
    ]
).astype("float32")
green_traces += 0.02 * rng.standard_normal(green_traces.shape).astype(
    "float32"
)
green_traces.shape

## 2. Wrap the array in a segment, attach to a recording

The recording owns metadata (fiber IDs, color, sampling rate). The segment owns the actual numpy array. `add_segment` wires them together &mdash; you can attach more than one segment for multi-block experiments.

In [ ]:
green_rec = BaseFiberPhotometryExtractor(
    sampling_frequency=sampling_rate,
    fiber_ids=fiber_ids,
    color="green",
    dtype="float32",
)

green_seg = NumpyFiberPhotometrySegment(
    traces=green_traces,
    sampling_frequency=sampling_rate,
)
green_rec.add_segment(green_seg)

green_rec

## 3. Read fluorescence

`get_fluorescence` is the fiber-photometry-native API. It accepts `segment_index`, `start_frame`, `end_frame`, and `fiber_ids` &mdash; no ephys-only flags like `return_in_uV`. Internally it delegates to spikeinterface's `get_traces`, so saving, slicing, and concatenation still work.

In [ ]:
# all fibers, full segment -> (n_samples, n_fibers)
all_traces = green_rec.get_fluorescence()
all_traces.shape

In [ ]:
# subset of fibers by ID
nac_only = green_rec.get_fluorescence(fiber_ids=["NAc"])
nac_only.shape

In [ ]:
# first second of data, all fibers
first_second = green_rec.get_fluorescence(start_frame=0, end_frame=100)
first_second.shape

## 4. Plot

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
for i, fid in enumerate(green_rec.fiber_ids):
    ax.plot(t, all_traces[:, i], label=fid)
ax.set_xlabel("time (s)")
ax.set_ylabel("fluorescence (a.u.)")
ax.set_title(
    f"{green_rec.color} channel \u2014 "
    f"{green_rec.get_num_fibers()} fibers"
)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()

## 5. Multi-color experiments

A real fiber photometry session usually records several colors from the same fibers &mdash; e.g. a calcium indicator (green) plus an isosbestic control (415 nm). Each color is its own recording with its own timebase. `FiberPhotometryRecordingGroup` bundles them and validates that they share the same fibers in the same order.

In [ ]:
iso_traces = (
    1.0
    + 0.05 * np.sin(2 * np.pi * 0.3 * t)[:, None]
    + 0.02 * rng.standard_normal((n_samples, 3))
).astype("float32")

iso_rec = BaseFiberPhotometryExtractor(
    sampling_frequency=sampling_rate,
    fiber_ids=fiber_ids,
    color="iso",
    dtype="float32",
)
iso_rec.add_segment(
    NumpyFiberPhotometrySegment(
        traces=iso_traces, sampling_frequency=sampling_rate
    )
)

experiment = FiberPhotometryRecordingGroup(
    {"green": green_rec, "iso": iso_rec}
)
experiment

## 6. Group access patterns

In [ ]:
experiment.colors, experiment.fiber_ids, experiment.get_num_fibers()

In [ ]:
# index by color, then call the per-color API
experiment["green"].get_fluorescence(fiber_ids=["NAc"]).shape

In [ ]:
# iterate over colors
for color, rec in experiment.items():
    print(f"{color:>6}: {rec.get_fluorescence().shape}")

## 7. Mismatched fibers fail loudly

The group enforces that every color shares the same fibers in the same order &mdash; downstream operations like dF/F or motion regression silently rely on this.

In [ ]:
bad = BaseFiberPhotometryExtractor(
    sampling_frequency=sampling_rate,
    fiber_ids=["VTA_left", "VTA_right"],  # missing NAc
    color="red",
    dtype="float32",
)
bad.add_segment(
    NumpyFiberPhotometrySegment(
        traces=np.zeros((n_samples, 2), dtype="float32"),
        sampling_frequency=sampling_rate,
    )
)

try:
    FiberPhotometryRecordingGroup({"green": green_rec, "red": bad})
except ValueError as exc:
    print("rejected:", exc)